In [ ]:
# pip install xgboost


  Using cached xgboost-3.4.1-py3-none-win_amd64.whl.metadata (2.0 kB)
Using cached xgboost-3.4.1-py3-none-win_amd64.whl (48.9 MB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [4]:
# 1. Zero-Safe Metric Functions
# Standard MAPE divides by actuals. If traffic density is 0, it explodes.
def wmape(y_true, y_pred):
    """Weighted Mean Absolute Percentage Error (Zero-Safe)"""
    return np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + 1e-8) * 100

def smape(y_true, y_pred):
    """Symmetric Mean Absolute Percentage Error (Zero-Safe)"""
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    return np.mean(np.abs(y_true - y_pred) / (denominator + 1e-8)) * 100

In [5]:
# 2. Load Data
INPUT_PATH = Path(r"C:\Users\KARTIK\OneDrive\Desktop\Master folder\Traffic flow project\data\processed\delhi_features.parquet")
df = pd.read_parquet(INPUT_PATH)

target_col = "Approach1_Queue"
features = [col for col in df.columns if col != target_col]

X = df[features]
y = df[target_col]

In [6]:
# 3. Chronological Train/Test Split (70/30)
# We must split by time, not randomly!
split_idx = int(len(df) * 0.7)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
print(f"Training on {len(X_train)} samples, Testing on {len(X_test)} samples")



Training on 1638 samples, Testing on 702 samples


In [13]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [16]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "XGBoost": xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1),
}

In [17]:
# 5. Train and Evaluate
results = []
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    # Calculate Metrics
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    w_mape = wmape(y_test, preds)
    s_mape = smape(y_test, preds)
    
    results.append({
        "Model": name,
        "MAE": round(mae, 4),
        "RMSE": round(rmse, 4),
        "WMAPE (%)": round(w_mape, 2),
        "sMAPE (%)": round(s_mape, 2)
    })

Training Linear Regression...
Training Ridge Regression...
Training Random Forest...
Training XGBoost...


In [18]:
# 6. Display Results Table
results_df = pd.DataFrame(results)
print("\n--- Baseline Evaluation Results (Stage 1) ---")
print(results_df.to_string(index=False))


--- Baseline Evaluation Results (Stage 1) ---
            Model    MAE   RMSE  WMAPE (%)  sMAPE (%)
Linear Regression 0.0000 0.0000       0.00       0.00
 Ridge Regression 0.0085 0.0127       1.65       2.01
    Random Forest 0.0163 0.0250       3.16       3.57
          XGBoost 0.0145 0.0231       2.81       3.14
